# Logical Fallacy Detection
**Dataset:** CoCoLoFa (Comments with Common Logical Fallacies)
https://github.com/Crowd-AI-Lab/cocolofa

**Pipeline:** Teks Mentah → Cleaning → Tokenisasi → Lemmatization → Representasi Teks (TF-IDF & Word2Vec)

## 0. Import Libraries

In [3]:
import json
import re
import numpy as np
import pandas as pd

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

print("All imports successful!")

All imports successful!


## 1. Load Dataset

In [4]:
# Load the train.json dataset
with open('train.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Extract all comments from every news article
records = []
for article in raw_data:
    for comment in article['comments']:
        records.append({
            'comment_id': comment['id'],
            'news_id': comment['news_id'],
            'fallacy': comment['fallacy'],
            'comment': comment['comment']
        })

df = pd.DataFrame(records)

print(f"Total komentar: {len(df)}")
print(f"\nDistribusi label fallacy:")
print(df['fallacy'].value_counts())
print(f"\n--- Sample Data (Teks Mentah) ---")
df.head(10)

Total komentar: 5370

Distribusi label fallacy:
fallacy
none                        2202
slippery slope               431
appeal to worse problems     421
appeal to nature             412
appeal to tradition          401
false dilemma                391
appeal to majority           383
hasty generalization         379
appeal to authority          350
Name: count, dtype: int64

--- Sample Data (Teks Mentah) ---


,comment_id,news_id,fallacy,comment
0,5584,262,none,Lack of transparency in government isn't unexp...
1,5582,262,appeal to authority,While the issues discussed here should be addr...
2,5583,262,none,The excuse that Brazilian municipalities do no...
3,6183,262,none,This is what's to be expected of developing an...
4,6182,262,appeal to tradition,"Sad to say, I have to agree with you. Rulers c..."
5,6184,262,none,Why doesn't the Brazilian federal government s...
6,6782,262,appeal to worse problems,At least they have a set up for it. In some co...
7,6783,262,hasty generalization,If these people can't even manage doing someth...
8,6784,262,appeal to tradition,I completely agree with you! There have alway...
9,10082,262,slippery slope,The argument that people should have access to...


## 2. Text Cleaning

In [5]:
def clean_text(text):
    """Membersihkan teks mentah."""
    text = text.lower()                                    # lowercase
    text = re.sub(r'http\S+|www\.\S+', '', text)           # hapus URL
    text = re.sub(r'<[^>]+>', '', text)                    # hapus HTML tags
    text = re.sub(r'@\w+', '', text)                       # hapus mentions
    text = re.sub(r'#\w+', '', text)                       # hapus hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)                # hapus angka & special chars
    text = re.sub(r'\s+', ' ', text).strip()               # hapus spasi berlebih
    return text

df['cleaned'] = df['comment'].apply(clean_text)

print("--- Contoh Sebelum & Sesudah Cleaning ---")
for i in range(3):
    print(f"\n[BEFORE]: {df['comment'].iloc[i][:120]}...")
    print(f"[AFTER ]: {df['cleaned'].iloc[i][:120]}...")

--- Contoh Sebelum & Sesudah Cleaning ---

[BEFORE]: Lack of transparency in government isn't unexpected.  No one likes to be under scrutiny,  especially not those with powe...
[AFTER ]: lack of transparency in government isnt unexpected no one likes to be under scrutiny especially not those with power tra...

[BEFORE]: While the issues discussed here should be addressed, there need to be priorities. The authorities of the country probabl...
[AFTER ]: while the issues discussed here should be addressed there need to be priorities the authorities of the country probably ...

[BEFORE]: The excuse that Brazilian municipalities do not have the funds to handle the public service requests does not hold water...
[AFTER ]: the excuse that brazilian municipalities do not have the funds to handle the public service requests does not hold water...


## 3. Tokenisasi

In [6]:
df['tokens'] = df['cleaned'].apply(word_tokenize)

print("--- Contoh Hasil Tokenisasi ---")
for i in range(3):
    print(f"\n[CLEANED ]: {df['cleaned'].iloc[i][:100]}...")
    print(f"[TOKENS  ]: {df['tokens'].iloc[i][:15]}...")
    print(f"Jumlah token: {len(df['tokens'].iloc[i])}")

--- Contoh Hasil Tokenisasi ---

[CLEANED ]: lack of transparency in government isnt unexpected no one likes to be under scrutiny especially not ...
[TOKENS  ]: ['lack', 'of', 'transparency', 'in', 'government', 'isnt', 'unexpected', 'no', 'one', 'likes', 'to', 'be', 'under', 'scrutiny', 'especially']...
Jumlah token: 64

[CLEANED ]: while the issues discussed here should be addressed there need to be priorities the authorities of t...
[TOKENS  ]: ['while', 'the', 'issues', 'discussed', 'here', 'should', 'be', 'addressed', 'there', 'need', 'to', 'be', 'priorities', 'the', 'authorities']...
Jumlah token: 55

[CLEANED ]: the excuse that brazilian municipalities do not have the funds to handle the public service requests...
[TOKENS  ]: ['the', 'excuse', 'that', 'brazilian', 'municipalities', 'do', 'not', 'have', 'the', 'funds', 'to', 'handle', 'the', 'public', 'service']...
Jumlah token: 93


## 4. Stopword Removal & Lemmatization

> Dipilih **lemmatization** daripada stemming karena menghasilkan kata dasar yang valid secara bahasa, lebih cocok untuk tugas deteksi fallacy yang membutuhkan pemahaman makna.

In [7]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    """Hapus stopwords dan lemmatize setiap token."""
    return [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2  # juga hapus token terlalu pendek
    ]

df['lemmatized'] = df['tokens'].apply(lemmatize_tokens)

# Gabungkan kembali menjadi string untuk keperluan TF-IDF
df['processed_text'] = df['lemmatized'].apply(lambda x: ' '.join(x))

print("--- Contoh Hasil Lemmatization ---")
for i in range(3):
    print(f"\n[TOKENS     ]: {df['tokens'].iloc[i][:12]}...")
    print(f"[LEMMATIZED ]: {df['lemmatized'].iloc[i][:12]}...")
    print(f"Jumlah token sebelum: {len(df['tokens'].iloc[i])} → sesudah: {len(df['lemmatized'].iloc[i])}")

--- Contoh Hasil Lemmatization ---

[TOKENS     ]: ['lack', 'of', 'transparency', 'in', 'government', 'isnt', 'unexpected', 'no', 'one', 'likes', 'to', 'be']...
[LEMMATIZED ]: ['lack', 'transparency', 'government', 'isnt', 'unexpected', 'one', 'like', 'scrutiny', 'especially', 'power', 'transparency', 'law']...
Jumlah token sebelum: 64 → sesudah: 31

[TOKENS     ]: ['while', 'the', 'issues', 'discussed', 'here', 'should', 'be', 'addressed', 'there', 'need', 'to', 'be']...
[LEMMATIZED ]: ['issue', 'discussed', 'addressed', 'need', 'priority', 'authority', 'country', 'probably', 'know', 'whats', 'important', 'deal']...
Jumlah token sebelum: 55 → sesudah: 29

[TOKENS     ]: ['the', 'excuse', 'that', 'brazilian', 'municipalities', 'do', 'not', 'have', 'the', 'funds', 'to', 'handle']...
[LEMMATIZED ]: ['excuse', 'brazilian', 'municipality', 'fund', 'handle', 'public', 'service', 'request', 'hold', 'water', 'evident', 'fact']...
Jumlah token sebelum: 93 → sesudah: 48


## 5. Representasi Teks

### a. TF-IDF (Term Frequency–Inverse Document Frequency) 

**TF-IDF** mengukur seberapa penting suatu kata dalam sebuah dokumen relatif terhadap seluruh korpus.
- Cocok sebagai baseline karena menangkap pola kata kunci yang sering muncul di tiap jenis fallacy.
- Menghasilkan sparse matrix berdimensi tinggi.

In [8]:
# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} dokumen, {tfidf_matrix.shape[1]} fitur")

# Tampilkan top fitur TF-IDF untuk beberapa sampel
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\nContoh 20 fitur (kata/bigram): {list(feature_names[:20])}")

# Konversi ke DataFrame untuk preview
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names
)
print(f"\n--- Preview TF-IDF Matrix (5 dokumen pertama, 10 fitur pertama) ---")
tfidf_df.iloc[:5, :10]

TF-IDF Matrix Shape: (5370, 5000)
  → 5370 dokumen, 5000 fitur

Contoh 20 fitur (kata/bigram): ['abandon', 'abandoned', 'ability', 'able', 'able access', 'able express', 'able get', 'able speak', 'absolute', 'absolutely', 'absurd', 'abuse', 'abuse power', 'abused', 'academic', 'academic freedom', 'accept', 'acceptable', 'acceptance', 'accepted']

--- Preview TF-IDF Matrix (5 dokumen pertama, 10 fitur pertama) ---


,abandon,abandoned,ability,able,able access,able express,able get,able speak,absolute,absolutely
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### b. Word2Vec
**Word2Vec** mempelajari embedding vektor untuk setiap kata berdasarkan konteks kemunculannya.
- Menangkap hubungan semantik antar kata (misal: "authority" dekat dengan "expert", "tradition" dekat dengan "culture").
- Representasi dokumen diperoleh dengan merata-ratakan vektor semua kata dalam dokumen.
- Menghasilkan dense vector berdimensi tetap untuk setiap dokumen.

In [9]:
# Train Word2Vec pada corpus lemmatized
w2v_model = Word2Vec(
    sentences=df['lemmatized'].tolist(),
    vector_size=100,    # dimensi embedding
    window=5,           # context window
    min_count=2,        # abaikan kata yang muncul < 2 kali
    workers=4,
    epochs=20,
    sg=1                # skip-gram (lebih baik untuk dataset kecil)
)

print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Vector dimension: {w2v_model.wv.vector_size}")

# Contoh kata-kata mirip
for word in ['government', 'freedom', 'protest']:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        print(f"\nKata mirip dengan '{word}':")
        for w, score in similar:
            print(f"  {w}: {score:.4f}")

Vocabulary size: 6642
Vector dimension: 100

Kata mirip dengan 'government':
  outrageous: 0.6433
  seizing: 0.6385
  subjugated: 0.6321
  fearful: 0.6293
  loses: 0.6257

Kata mirip dengan 'freedom':
  expression: 0.7130
  inviolate: 0.6163
  speech: 0.6090
  free: 0.6090
  stripped: 0.6049

Kata mirip dengan 'protest':
  demonstration: 0.5822
  peaceful: 0.5528
  protesting: 0.5380
  squash: 0.5296
  organize: 0.5258


In [10]:
def document_vector(tokens, model):
    """Hitung rata-rata Word2Vec vector untuk seluruh token dalam dokumen."""
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

# Buat document vectors untuk semua komentar
w2v_vectors = np.array([
    document_vector(tokens, w2v_model) for tokens in df['lemmatized']
])

w2v_df = pd.DataFrame(
    w2v_vectors,
    columns=[f'w2v_dim_{i}' for i in range(w2v_model.wv.vector_size)]
)

print(f"Word2Vec Document Matrix Shape: {w2v_vectors.shape}")
print(f"  → {w2v_vectors.shape[0]} dokumen, {w2v_vectors.shape[1]} dimensi")
print(f"\n--- Preview Word2Vec Document Vectors (5 dokumen pertama, 10 dimensi pertama) ---")
w2v_df.iloc[:5, :10]

Word2Vec Document Matrix Shape: (5370, 100)
  → 5370 dokumen, 100 dimensi

--- Preview Word2Vec Document Vectors (5 dokumen pertama, 10 dimensi pertama) ---


,w2v_dim_0,w2v_dim_1,w2v_dim_2,w2v_dim_3,w2v_dim_4,w2v_dim_5,w2v_dim_6,w2v_dim_7,w2v_dim_8,w2v_dim_9
0,-0.091344,0.080455,0.006898,0.001491,0.064281,-0.402619,0.058436,0.307599,-0.120050,-0.188737
1,-0.183424,0.022875,0.180860,-0.013968,0.134161,-0.181686,0.202886,0.256813,-0.008195,-0.179281
2,-0.163573,0.175520,0.035289,0.013282,0.145032,-0.322679,0.060238,0.337482,-0.141012,-0.049891
3,-0.075951,0.220182,-0.008100,0.010337,-0.015438,-0.310571,0.165169,0.310540,-0.091039,-0.183326
4,-0.201556,0.211644,0.115100,-0.020554,-0.047806,-0.286604,0.153015,0.306859,-0.110499,-0.224565
